In [1]:
print(5)

5


# TFLite Quantization — 4 MobileNetV2 Models

Converts 4 fine-tuned `.h5` models to TFLite in **float32**, **float16**, and **INT8** formats,
then benchmarks file size, accuracy, and inference time.

| Model | Type | Output |
|-------|------|--------|
| Cracking | Binary | sigmoid (threshold 0.5) |
| Stringing | Binary | sigmoid (threshold 0.5) |
| Warping | Binary | sigmoid (threshold 0.5) |
| 4-Class | Multi-class | softmax (argmax) |

> Preprocessing: resize to 224×224, normalize to [0, 1]

---
## 1. Imports & Configuration

In [2]:
import os, glob, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)
import matplotlib.pyplot as plt

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

BASE    = os.getcwd()
DATA    = os.path.join(BASE, "data")
TFLITE  = os.path.join(BASE, "tflite_models")
os.makedirs(TFLITE, exist_ok=True)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
N_CALIB    = 150   # images used for INT8 calibration
N_BENCH    = 100   # images used for timing benchmark
SEED       = 42


2026-05-16 09:05:26.525677: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778918726.623704    1063 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778918726.651096    1063 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-16 09:05:26.874055: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow: 2.18.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


---
## 2. Helper Functions

In [3]:
# ── Image loading ──────────────────────────────────────────────────────────────
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img.numpy()

def load_images_from_dir(folder, max_images=None):
    paths = sorted(glob.glob(os.path.join(folder, "*.jpg")))
    if max_images:
        paths = paths[:max_images]
    return np.array([load_image(p) for p in paths], dtype=np.float32)

# ── File size ──────────────────────────────────────────────────────────────────
def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)

# ── TFLite interpreter ─────────────────────────────────────────────────────────
def get_tflite_interpreter(tflite_path):
    interp = tf.lite.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    return interp

def tflite_predict_single(interp, image_np):
    """Run inference on one (H, W, 3) image; returns raw output array."""
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    img = image_np[np.newaxis].astype(inp["dtype"])
    interp.set_tensor(inp["index"], img)
    interp.invoke()
    return interp.get_tensor(out["index"])

# ── Evaluate TFLite (binary) ───────────────────────────────────────────────────
def evaluate_tflite_binary(tflite_path, images, labels, threshold=0.5):
    interp = get_tflite_interpreter(tflite_path)
    preds = []
    for img in images:
        out = tflite_predict_single(interp, img)
        preds.append(int(out.ravel()[0] >= threshold))
    preds = np.array(preds)
    return dict(
        accuracy  = accuracy_score(labels, preds),
        precision = precision_score(labels, preds, zero_division=0),
        recall    = recall_score(labels, preds, zero_division=0),
        f1        = f1_score(labels, preds, zero_division=0),
    )

# ── Evaluate TFLite (4-class) ──────────────────────────────────────────────────
def evaluate_tflite_multiclass(tflite_path, images, labels, n_classes=4):
    interp = get_tflite_interpreter(tflite_path)
    preds = []
    for img in images:
        out = tflite_predict_single(interp, img)
        preds.append(int(np.argmax(out.ravel())))
    preds = np.array(preds)
    return dict(
        accuracy  = accuracy_score(labels, preds),
        precision = precision_score(labels, preds, average="macro", zero_division=0),
        recall    = recall_score(labels, preds, average="macro", zero_division=0),
        f1        = f1_score(labels, preds, average="macro", zero_division=0),
    )

# ── Benchmark inference time ───────────────────────────────────────────────────
def benchmark_tflite(tflite_path, images, n=N_BENCH):
    interp = get_tflite_interpreter(tflite_path)
    imgs   = images[:n]
    times  = []
    for img in imgs:
        t0 = time.time()
        tflite_predict_single(interp, img)
        times.append((time.time() - t0) * 1000)   # ms
    return np.mean(times)

def benchmark_keras(model, images, n=N_BENCH):
    imgs  = images[:n]
    times = []
    for img in imgs:
        t0 = time.time()
        model.predict(img[np.newaxis], verbose=0)
        times.append((time.time() - t0) * 1000)
    return np.mean(times)

print("✅ Helper functions ready.")


✅ Helper functions ready.


---
## 3. Load Test Datasets

In [4]:
# ── Binary test sets (use all images in each folder) ──────────────────────────
print("Loading binary test data...")

imgs_cracking    = load_images_from_dir(os.path.join(DATA, "Cracking"))
imgs_no_cracking = load_images_from_dir(os.path.join(DATA, "No_Cracking"))
X_crack = np.concatenate([imgs_cracking, imgs_no_cracking])
y_crack = np.array([1]*len(imgs_cracking) + [0]*len(imgs_no_cracking))
print(f"  Cracking   : {len(imgs_cracking)} pos + {len(imgs_no_cracking)} neg = {len(X_crack)} total")



Loading binary test data...


I0000 00:00:1778918738.126258    1063 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6100 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060 SUPER, pci bus id: 0000:01:00.0, compute capability: 7.5


  Cracking   : 1663 pos + 2042 neg = 3705 total


In [ ]:
imgs_stringing    = load_images_from_dir(os.path.join(DATA, "Stringing"))
imgs_no_stringing = load_images_from_dir(os.path.join(DATA, "No_Stringing"))
X_string = np.concatenate([imgs_stringing, imgs_no_stringing])
y_string = np.array([1]*len(imgs_stringing) + [0]*len(imgs_no_stringing))
print(f"  Stringing  : {len(imgs_stringing)} pos + {len(imgs_no_stringing)} neg = {len(X_string)} total")

In [ ]:
imgs_warping    = load_images_from_dir(os.path.join(DATA, "Warping"))
imgs_no_warping = load_images_from_dir(os.path.join(DATA, "No_Warping"))
X_warp = np.concatenate([imgs_warping, imgs_no_warping])
y_warp = np.array([1]*len(imgs_warping) + [0]*len(imgs_no_warping))
print(f"  Warping    : {len(imgs_warping)} pos + {len(imgs_no_warping)} neg = {len(X_warp)} total")

In [ ]:
# ── 4-class test set (same split as training: 10% test, SEED=42) ──────────────
print("\nLoading 4-class test data...")
CLASS_FOLDERS_4 = [
    ("Cracking",    0),
    ("Stringing",   1),
    ("Warping",     2),
    ("No_Stringing",3),   # No_Defect
]
all_paths4, all_labels4 = [], []
for folder, lbl in CLASS_FOLDERS_4:
    paths = sorted(glob.glob(os.path.join(DATA, folder, "*.jpg")))
    all_paths4.extend(paths)
    all_labels4.extend([lbl]*len(paths))
    print(f"  {folder:<15}: {len(paths)} images  →  label {lbl}")

In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(all_paths4))
all_paths4  = np.array(all_paths4)[idx]
all_labels4 = np.array(all_labels4)[idx]
n = len(all_paths4)
test_paths4  = all_paths4[int(n*0.90):]
test_labels4 = all_labels4[int(n*0.90):]
X_4class = np.array([load_image(p) for p in test_paths4], dtype=np.float32)
y_4class = test_labels4
print(f"\n4-class test set: {len(X_4class)} images")

---
## 4. TFLite Conversion Functions

In [ ]:
def convert_float32(keras_model, out_path):
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    tflite_model = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    print(f"  [float32] → {out_path}  ({file_size_mb(out_path):.2f} MB)")
    return out_path

def convert_float16(keras_model, out_path):
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_model = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    print(f"  [float16] → {out_path}  ({file_size_mb(out_path):.2f} MB)")
    return out_path

def convert_int8(keras_model, out_path, calib_images):
    def representative_dataset():
        for img in calib_images:
            yield [img[np.newaxis].astype(np.float32)]
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type  = tf.int8
    converter.inference_output_type = tf.int8
    tflite_model = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    print(f"  [int8]    → {out_path}  ({file_size_mb(out_path):.2f} MB)")
    return out_path

print("✅ Conversion functions ready.")


---
## 5. Define Model Configurations

In [ ]:
MODELS = [
    dict(
        name    = "Cracking",
        h5      = os.path.join(BASE, "cracking", "mobilenetv2_cracking_finetuned_final.h5"),
        mode    = "binary",
        X_test  = None,   # filled below
        y_test  = None,
        X_calib = None,
    ),
    dict(
        name    = "Stringing",
        h5      = os.path.join(BASE, "Stringing", "mobilenetv2_stringing_finetuned_final.h5"),
        mode    = "binary",
        X_test  = None,
        y_test  = None,
        X_calib = None,
    ),
    dict(
        name    = "Warping",
        h5      = os.path.join(BASE, "warping", "mobilenetv2_Warping_finetuned_final.h5"),
        mode    = "binary",
        X_test  = None,
        y_test  = None,
        X_calib = None,
    ),
    dict(
        name    = "4-Class",
        h5      = os.path.join(BASE, "mobilenetv2_4class_alllayers_final.h5"),
        mode    = "multiclass",
        X_test  = None,
        y_test  = None,
        X_calib = None,
    ),
]

# Attach test / calib arrays
MODELS[0]["X_test"]  = X_crack;   MODELS[0]["y_test"]  = y_crack
MODELS[1]["X_test"]  = X_string;  MODELS[1]["y_test"]  = y_string
MODELS[2]["X_test"]  = X_warp;    MODELS[2]["y_test"]  = y_warp
MODELS[3]["X_test"]  = X_4class;  MODELS[3]["y_test"]  = y_4class

# Calibration images (N_CALIB from positive class of each binary set)
MODELS[0]["X_calib"] = imgs_cracking[:N_CALIB]
MODELS[1]["X_calib"] = imgs_stringing[:N_CALIB]
MODELS[2]["X_calib"] = imgs_warping[:N_CALIB]
MODELS[3]["X_calib"] = X_4class[:N_CALIB]

for m in MODELS:
    print(f"  {m['name']:<12}  test={len(m['X_test'])}  calib={len(m['X_calib'])}  h5_exists={os.path.exists(m['h5'])}")


---
## 6. Convert, Evaluate & Benchmark All Models

For each model we:
1. Convert to **float32**, **float16**, and **INT8** TFLite
2. Evaluate accuracy / precision / recall / F1 on the test set
3. Measure average inference time over the first 100 images

In [ ]:
results = []   # one dict per (model, format)

for cfg in MODELS:
    name     = cfg["name"]
    mode     = cfg["mode"]
    h5_path  = cfg["h5"]
    X_test   = cfg["X_test"]
    y_test   = cfg["y_test"]
    X_calib  = cfg["X_calib"]
    prefix   = name.lower().replace("-", "_")

    print(f"\n{'='*60}")
    print(f"  MODEL: {name}")
    print(f"{'='*60}")

    # -- Load Keras model --------------------------------------------------
    print("  Loading Keras model…")
    model = keras.models.load_model(h5_path)
    h5_size = file_size_mb(h5_path)

    # -- Keras baseline timing --------------------------------------------
    print("  Benchmarking Keras inference…")
    keras_time = benchmark_keras(model, X_test)

    # -- Keras accuracy ----------------------------------------------------
    print("  Evaluating Keras accuracy…")
    if mode == "binary":
        keras_preds_raw = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
        keras_preds = (keras_preds_raw >= 0.5).astype(int)
    else:
        keras_preds_raw = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
        keras_preds = np.argmax(keras_preds_raw, axis=1)

    avg = "binary" if mode == "binary" else "macro"
    keras_metrics = dict(
        accuracy  = accuracy_score(y_test, keras_preds),
        precision = precision_score(y_test, keras_preds, average=avg, zero_division=0),
        recall    = recall_score(y_test, keras_preds, average=avg, zero_division=0),
        f1        = f1_score(y_test, keras_preds, average=avg, zero_division=0),
    )
    results.append({"model": name, "format": "Keras (.h5)",
                    "size_mb": h5_size, "inf_ms": keras_time, **keras_metrics})
    print(f"  Keras  acc={keras_metrics['accuracy']:.4f}  f1={keras_metrics['f1']:.4f}  size={h5_size:.2f} MB  time={keras_time:.1f} ms")

    # ── float32 ────────────────────────────────────────────────────────────
    f32_path = os.path.join(TFLITE, f"{prefix}_float32.tflite")
    print("  Converting → float32 TFLite…")
    convert_float32(model, f32_path)
    print("  Evaluating float32…")
    if mode == "binary":
        m32 = evaluate_tflite_binary(f32_path, X_test, y_test)
    else:
        m32 = evaluate_tflite_multiclass(f32_path, X_test, y_test)
    t32 = benchmark_tflite(f32_path, X_test)
    results.append({"model": name, "format": "TFLite float32",
                    "size_mb": file_size_mb(f32_path), "inf_ms": t32, **m32})
    print(f"  float32 acc={m32['accuracy']:.4f}  f1={m32['f1']:.4f}  size={file_size_mb(f32_path):.2f} MB  time={t32:.1f} ms")

    # ── float16 ────────────────────────────────────────────────────────────
    f16_path = os.path.join(TFLITE, f"{prefix}_float16.tflite")
    print("  Converting → float16 TFLite…")
    convert_float16(model, f16_path)
    print("  Evaluating float16…")
    if mode == "binary":
        m16 = evaluate_tflite_binary(f16_path, X_test, y_test)
    else:
        m16 = evaluate_tflite_multiclass(f16_path, X_test, y_test)
    t16 = benchmark_tflite(f16_path, X_test)
    results.append({"model": name, "format": "TFLite float16",
                    "size_mb": file_size_mb(f16_path), "inf_ms": t16, **m16})
    print(f"  float16 acc={m16['accuracy']:.4f}  f1={m16['f1']:.4f}  size={file_size_mb(f16_path):.2f} MB  time={t16:.1f} ms")

    # ── INT8 ───────────────────────────────────────────────────────────────
    i8_path = os.path.join(TFLITE, f"{prefix}_int8.tflite")
    print("  Converting → INT8 TFLite (calibrating)…")
    convert_int8(model, i8_path, X_calib)
    print("  Evaluating INT8…")
    # INT8 models have int8 I/O; use a wrapper that handles dtype
    interp_i8 = get_tflite_interpreter(i8_path)
    inp_detail = interp_i8.get_input_details()[0]
    scale, zp  = inp_detail["quantization"]

    def tflite_predict_int8(img_np):
        interp = get_tflite_interpreter(i8_path)
        inp_d  = interp.get_input_details()[0]
        out_d  = interp.get_output_details()[0]
        s, z   = inp_d["quantization"]
        img_q  = (img_np / s + z).astype(np.int8)
        interp.set_tensor(inp_d["index"], img_q[np.newaxis])
        interp.invoke()
        raw = interp.get_tensor(out_d["index"])
        # dequantize output
        os2, oz2 = out_d["quantization"]
        return (raw.astype(np.float32) - oz2) * os2

    preds_i8 = []
    for img in X_test:
        out = tflite_predict_int8(img)
        if mode == "binary":
            preds_i8.append(int(out.ravel()[0] >= 0.5))
        else:
            preds_i8.append(int(np.argmax(out.ravel())))
    preds_i8 = np.array(preds_i8)

    mi8 = dict(
        accuracy  = accuracy_score(y_test, preds_i8),
        precision = precision_score(y_test, preds_i8, average=avg, zero_division=0),
        recall    = recall_score(y_test, preds_i8, average=avg, zero_division=0),
        f1        = f1_score(y_test, preds_i8, average=avg, zero_division=0),
    )
    # timing for INT8
    t0s = []
    for img in X_test[:N_BENCH]:
        t0 = time.time()
        tflite_predict_int8(img)
        t0s.append((time.time() - t0)*1000)
    ti8 = np.mean(t0s)

    results.append({"model": name, "format": "TFLite INT8",
                    "size_mb": file_size_mb(i8_path), "inf_ms": ti8, **mi8})
    print(f"  INT8    acc={mi8['accuracy']:.4f}  f1={mi8['f1']:.4f}  size={file_size_mb(i8_path):.2f} MB  time={ti8:.1f} ms")

    del model   # free GPU memory

print("\n✅ All models converted and evaluated.")


---
## 7. Comparison Table

In [ ]:
df = pd.DataFrame(results)
df["size_mb"]  = df["size_mb"].round(2)
df["inf_ms"]   = df["inf_ms"].round(2)
df["accuracy"] = (df["accuracy"]*100).round(2)
df["f1"]       = (df["f1"]*100).round(2)
df["precision"]= (df["precision"]*100).round(2)
df["recall"]   = (df["recall"]*100).round(2)

df = df.rename(columns={
    "model":    "Model",
    "format":   "Format",
    "size_mb":  "Size (MB)",
    "inf_ms":   "Inference (ms)",
    "accuracy": "Accuracy (%)",
    "precision":"Precision (%)",
    "recall":   "Recall (%)",
    "f1":       "F1 (%)",
})
display(df[["Model","Format","Size (MB)","Accuracy (%)","Precision (%)","Recall (%)","F1 (%)","Inference (ms)"]])
df.to_csv("quantization_results.csv", index=False)
print("✅ Results saved to quantization_results.csv")


---
## 8. Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

formats   = ["Keras (.h5)", "TFLite float32", "TFLite float16", "TFLite INT8"]
model_names = ["Cracking", "Stringing", "Warping", "4-Class"]
colors    = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for ax, metric in zip(axes, ["Accuracy (%)", "Size (MB)", "Inference (ms)"]):
    x = np.arange(len(formats))
    width = 0.18
    for i, mname in enumerate(model_names):
        sub = df[df["Model"] == mname]
        vals = [sub[sub["Format"]==f][metric].values[0] if len(sub[sub["Format"]==f]) else 0 for f in formats]
        ax.bar(x + i*width, vals, width, label=mname, color=colors[i], alpha=0.85)
    ax.set_title(metric)
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(formats, rotation=15, ha="right", fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("TFLite Quantization — Accuracy / Size / Speed", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
